# Implementing GPT using Hugging Face

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [3]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [24]:
prompt = "Artitificial Intelligence is"
prompt = "In the year 2050, humans discovered that"

inputs = tokenizer(prompt, return_tensors='pt').to(device)

output = model.generate(
    **inputs,
    max_length=50
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In the year 2050, humans discovered that the Earth's surface is covered with a layer of carbon dioxide, which is a greenhouse gas. The carbon dioxide is released into the atmosphere by the burning of fossil fuels.

The carbon dioxide is released into


## different decoding strategies

In [44]:
output2 = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,  #greedy sampling
    temperature=1.2, #controls randomness by scaling logits before softmax
    top_k=50,        #keeps only top k most probable tokens
    top_p=0.9,        #nucleus sampling
    num_return_sequences=3, #generates multiple outputs
    #num_beams=5,        #beam search (often used for translation/summarization)
)

out1 = tokenizer.decode(output)
out2 = tokenizer.decode(output2[0], skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


| Temperature | Behavior              |
| ----------- | --------------------- |
| < 1.0       | Conservative, safer   |
| = 1.0       | Normal                |
| > 1.0       | More creative, random |


In [45]:
print(out1)
print(out2)

["In the year 2050, humans discovered that the Earth's surface is covered with a layer of carbon dioxide, which is a greenhouse gas. The carbon dioxide is released into the atmosphere by the burning of fossil fuels.\n\nThe carbon dioxide is released into"]
In the year 2050, humans discovered that the sun's energy had almost completely evaporated by then and no longer existed, forcing astronomers to look at just how massive it had grown. One new study from Cornell University suggests it might take up to a billion


In [46]:
print(tokenizer.decode(output2[0], skip_special_tokens=True))
print('-------------------------------------------------------------')
print(tokenizer.decode(output2[1], skip_special_tokens=True))
print('-------------------------------------------------------------')
print(tokenizer.decode(output2[2], skip_special_tokens=True))

In the year 2050, humans discovered that the sun's energy had almost completely evaporated by then and no longer existed, forcing astronomers to look at just how massive it had grown. One new study from Cornell University suggests it might take up to a billion
-------------------------------------------------------------
In the year 2050, humans discovered that more than 60% of the solar energy in our homes was from hydrocarbons, and we will now see the beginning of a global catastrophe that could destroy the planet.
-------------------------------------------------------------
In the year 2050, humans discovered that Neanderthals had long been out of Africa. This discovery, which has implications for human evolution for about 100 million years, means that our modern population may have moved farther across the African plateau and thus farther out


In [53]:

prompt = "By 2030, most of the youth population in India"

inputs = tokenizer(prompt, return_tensors='pt').to(device)

output3 = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    temperature=1.2,
    num_return_sequences=2
)

print(tokenizer.decode(output3[0], skip_special_tokens=True))
print(tokenizer.decode(output3[1], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


By 2030, most of the youth population in India will be between nine and twelve, who account for over ten per cent of this population. This may lead us towards a major demographic crisis: over 80 per cent of India's population has already developed high
By 2030, most of the youth population in India have a very broad knowledge base, the rest can only speak rudimentary English or study foreign language (and sometimes English in some way). This is often just a fact of life for students in particular. While


## System Resource Monitoring

In [13]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA Available: False
GPU Name: No GPU


In [14]:
if torch.cuda.is_available():
    print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
    print("Reserved:", torch.cuda.memory_reserved()/1024**3, "GB")

In [15]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [16]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("Total Parameters:", count_parameters(model))

Total Parameters: 124439808


In [17]:
print(f"{count_parameters(model)/1e6:.2f} Million parameters")

124.44 Million parameters


In [18]:
import psutil

ram = psutil.virtual_memory()
print("Total RAM:", ram.total/1024**3, "GB")
print("Used RAM:", ram.used/1024**3, "GB")

Total RAM: 11.571647644042969 GB
Used RAM: 6.005836486816406 GB
